# Safety Solver/Judge Difficulty Comparison

Compare solver-side K-factor item difficulty from `safety_all_attacks` against judge-side continuous K-factor difficulty from `harmmetric_eval`.

Safety solver items are attack-expanded (`prompt x attack_family`), so this notebook first aggregates solver difficulty to the prompt level, then joins to HarmMetric prompt-level judging items.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 240)


## Choose K And Resolve Paths


In [ ]:
SOLVER_K = 2  # choose 1 or 2
JUDGE_K = 2   # choose 1 or 2


def find_repo_root(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "K-Factor").exists() and (candidate / "benchmarks").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing K-Factor/ and benchmarks/")


REPO_ROOT = find_repo_root()
print(f"Repo root: {REPO_ROOT}")

solver_json_path = REPO_ROOT / "K-Factor" / "results" / "safety_all_attacks" / "safety_all_attacks_kfactor_item_difficulties_with_laplace_uncertainty.json"
solver_fit_summary_path = REPO_ROOT / "K-Factor" / "results" / "safety_all_attacks" / "safety_all_attacks_kfactor_fit_summary.csv"

judge_json_path = REPO_ROOT / "K-Factor" / "results_continuous" / "harmmetric_eval" / "harmmetric_eval_continuous_kfactor_item_difficulties_with_laplace_uncertainty.json"
judge_fit_summary_path = REPO_ROOT / "K-Factor" / "results_continuous" / "harmmetric_eval" / "harmmetric_eval_continuous_kfactor_fit_summary.csv"

solver_scored_path = REPO_ROOT / "benchmarks" / "safety" / "final_solver" / "all_attacks_all_models_scored.json"
direct_scored_path = REPO_ROOT / "benchmarks" / "safety" / "final_solver" / "direct_all_models_scored.json"
harmmetric_prompt_scores_dir = REPO_ROOT / "benchmarks" / "HarmMetric_Eval" / "Final files to use"

out_dir = REPO_ROOT / "K-Factor" / "results" / "safety_solver_judge_comparison"
out_dir.mkdir(parents=True, exist_ok=True)

for path in [solver_json_path, solver_fit_summary_path, judge_json_path, judge_fit_summary_path, solver_scored_path, direct_scored_path, harmmetric_prompt_scores_dir]:
    print(path.relative_to(REPO_ROOT), path.exists())

if not judge_json_path.exists():
    print("Missing HarmMetric continuous K-factor results. Run K-Factor/kfactor_continuous.ipynb with KFACTOR_MATRIX='harmmetric_eval' first, then rerun this notebook.")


## Fit Summaries


In [ ]:
def choose_best_k(summary):
    score_col = "auc" if "auc" in summary.columns else None
    tie_cols = [c for c in ["ece", "loss", "beta_nll", "mse"] if c in summary.columns]
    if score_col is not None:
        return summary.sort_values([score_col, *tie_cols], ascending=[False, *([True] * len(tie_cols))]).iloc[0]
    return summary.sort_values(tie_cols or ["k"]).iloc[0]


solver_fit_summary = pd.read_csv(solver_fit_summary_path)
display(solver_fit_summary.round(4))
solver_best = choose_best_k(solver_fit_summary)
print(f"Solver-side best k: k={int(solver_best['k'])}")

if judge_fit_summary_path.exists():
    judge_fit_summary = pd.read_csv(judge_fit_summary_path)
    display(judge_fit_summary.round(4))
    judge_best = choose_best_k(judge_fit_summary)
    print(f"Judge-side best k:  k={int(judge_best['k'])}")
else:
    judge_fit_summary = pd.DataFrame()

print(f"Currently loading solver K={SOLVER_K}, judge K={JUDGE_K}")

solver_fit_summary.to_csv(out_dir / "safety_solver_kfactor_fit_summary.csv", index=False)
if not judge_fit_summary.empty:
    judge_fit_summary.to_csv(out_dir / "harmmetric_eval_continuous_kfactor_fit_summary.csv", index=False)


## Load K-Factor Item JSONs


In [ ]:
def load_kfactor_fit(json_path, k, score_col):
    with open(json_path) as f:
        payload = json.load(f)

    fit_key = f"k{k}"
    if fit_key not in payload["fits"]:
        raise KeyError(f"{json_path} does not contain {fit_key}; available={list(payload['fits'])}")

    df = pd.DataFrame(payload["fits"][fit_key])
    df["item_id"] = df["item_id"].astype(str)
    df = df.rename(columns={"scores": score_col})
    return df, payload


solver_attack_items, solver_payload = load_kfactor_fit(solver_json_path, SOLVER_K, "solver_scores")
print(f"solver_attack_items: {solver_attack_items.shape}")
display(solver_attack_items.head())

solver_attack_items.to_csv(out_dir / f"safety_solver_k{SOLVER_K}_attack_items.csv", index=False)
solver_attack_items.to_json(out_dir / f"safety_solver_k{SOLVER_K}_attack_items.json", orient="records", indent=2, force_ascii=False)

if judge_json_path.exists():
    judge_items, judge_payload = load_kfactor_fit(judge_json_path, JUDGE_K, "judge_scores")
    judge_items["prompt_id"] = judge_items["item_id"].astype(int)
    print(f"judge_items: {judge_items.shape}")
    display(judge_items.head())
    judge_items.to_csv(out_dir / f"harmmetric_eval_k{JUDGE_K}_judge_items.csv", index=False)
    judge_items.to_json(out_dir / f"harmmetric_eval_k{JUDGE_K}_judge_items.json", orient="records", indent=2, force_ascii=False)
else:
    judge_items = pd.DataFrame()


## Build Prompt ID Lookup


In [ ]:
# Solver side: use direct prompts as the canonical original prompt lookup.
# Attack-family harmful_prompt fields can be transformed prompts, so do not use them for HarmMetric matching.
with open(direct_scored_path) as f:
    direct_rows = json.load(f)

solver_prompt_lookup = (
    pd.DataFrame(direct_rows)[["input_index", "harmful_prompt", "source"]]
    .drop_duplicates()
    .rename(columns={"harmful_prompt": "original_prompt"})
    .copy()
)
solver_prompt_lookup["input_index"] = solver_prompt_lookup["input_index"].astype(int)

# Judge side: prompt_id -> original prompt/source from the HarmMetric final prompt-score files.
prompt_score_files = sorted(harmmetric_prompt_scores_dir.glob("*safety_solver_prompt_scores*.csv"))
harmmetric_prompt_lookup = pd.concat(
    [
        pd.read_csv(path, usecols=["prompt_id", "harmful_prompt", "source"])
        for path in prompt_score_files
    ],
    ignore_index=True,
).drop_duplicates()
harmmetric_prompt_lookup = harmmetric_prompt_lookup.rename(columns={"harmful_prompt": "original_prompt"})
harmmetric_prompt_lookup["prompt_id"] = harmmetric_prompt_lookup["prompt_id"].astype(int)

prompt_lookup = solver_prompt_lookup.merge(
    harmmetric_prompt_lookup,
    on=["original_prompt", "source"],
    how="left",
    validate="one_to_one",
)

print(f"solver_prompt_lookup from direct: {solver_prompt_lookup.shape}")
print(f"harmmetric_prompt_lookup:       {harmmetric_prompt_lookup.shape}")
print(f"prompt_lookup:                  {prompt_lookup.shape}")
print(f"missing prompt_id:              {prompt_lookup['prompt_id'].isna().sum()}")

display(prompt_lookup[["input_index", "source", "prompt_id"]].head())

prompt_lookup.to_csv(out_dir / "safety_input_index_to_harmmetric_prompt_id.csv", index=False)
prompt_lookup.to_json(out_dir / "safety_input_index_to_harmmetric_prompt_id.json", orient="records", indent=2, force_ascii=False)


## Aggregate Solver Difficulty To Prompt Level


In [ ]:
def combine_se_for_mean(values):
    vals = pd.Series(values).dropna().astype(float)
    if vals.empty:
        return np.nan
    return float(np.sqrt(np.square(vals).sum()) / len(vals))


def mean_score_dict(dicts):
    models = sorted({model for scores in dicts if isinstance(scores, dict) for model in scores})
    averaged = {}
    for model in models:
        vals = []
        for scores in dicts:
            if isinstance(scores, dict):
                val = scores.get(model)
                if val is not None and not pd.isna(val):
                    vals.append(float(val))
        averaged[model] = float(np.mean(vals)) if vals else None
    return averaged


solver_attack_items = solver_attack_items.merge(
    prompt_lookup[["input_index", "prompt_id", "original_prompt", "source"]],
    on=["input_index", "source"],
    how="left",
)
solver_attack_items["prompt_id"] = solver_attack_items["prompt_id"].astype("Int64")

solver_prompt_items = (
    solver_attack_items
    .dropna(subset=["prompt_id"])
    .groupby("prompt_id", as_index=False)
    .agg(
        harmful_prompt=("original_prompt", "first"),
        source=("source", "first"),
        n_attacks=("attack_family", "count"),
        attack_families=("attack_family", lambda s: sorted(s.astype(str).unique())),
        solver_difficulty_mean=("difficulty_centered", "mean"),
        solver_difficulty_min=("difficulty_centered", "min"),
        solver_difficulty_max=("difficulty_centered", "max"),
        solver_difficulty_sd_across_attacks=("difficulty_centered", "std"),
        solver_difficulty_laplace_se_mean=("difficulty_centered_laplace_se", combine_se_for_mean),
        solver_difficulty_laplace_se_avg=("difficulty_centered_laplace_se", "mean"),
        solver_scores=("solver_scores", mean_score_dict),
    )
)
solver_prompt_items["solver_difficulty_range_low"] = solver_prompt_items["solver_difficulty_mean"] - solver_prompt_items["solver_difficulty_min"]
solver_prompt_items["solver_difficulty_range_high"] = solver_prompt_items["solver_difficulty_max"] - solver_prompt_items["solver_difficulty_mean"]

print(f"solver_prompt_items: {solver_prompt_items.shape}")
display(solver_prompt_items.drop(columns=["harmful_prompt", "solver_scores"]).head())

solver_prompt_items.to_csv(out_dir / f"safety_solver_k{SOLVER_K}_prompt_items.csv", index=False)
solver_prompt_items.to_json(out_dir / f"safety_solver_k{SOLVER_K}_prompt_items.json", orient="records", indent=2, force_ascii=False)


## Join Solver Prompt Difficulty To Judge Difficulty


In [ ]:
if judge_items.empty:
    raise FileNotFoundError("HarmMetric continuous K-factor results are missing. Run K-Factor/kfactor_continuous.ipynb first.")

judge_for_join = judge_items.rename(columns={
    "difficulty_centered": "judge_difficulty_centered",
    "difficulty_centered_laplace_se": "judge_difficulty_centered_laplace_se",
    "difficulty": "judge_difficulty",
    "difficulty_laplace_se": "judge_difficulty_laplace_se",
})

paired_items = solver_prompt_items.merge(
    judge_for_join[[
        "prompt_id",
        "judge_difficulty",
        "judge_difficulty_centered",
        "judge_difficulty_laplace_se",
        "judge_difficulty_centered_laplace_se",
        "judge_scores",
    ]],
    on="prompt_id",
    how="inner",
    validate="one_to_one",
)

print(f"paired_items: {paired_items.shape}")
display(paired_items.head())

paired_items.to_csv(out_dir / f"safety_k{SOLVER_K}_solver_harmmetric_k{JUDGE_K}_paired_items.csv", index=False)
paired_items.to_json(out_dir / f"safety_k{SOLVER_K}_solver_harmmetric_k{JUDGE_K}_paired_items.json", orient="records", indent=2, force_ascii=False)


## Part One: Solver Prompt Difficulty vs Mean Judge Score


In [ ]:
partone_table = paired_items[[
    "prompt_id",
    "source",
    "harmful_prompt",
    "solver_difficulty_mean",
    "solver_difficulty_laplace_se_mean",
    "solver_difficulty_sd_across_attacks",
    "solver_difficulty_min",
    "solver_difficulty_max",
    "solver_scores",
    "judge_scores",
]].copy()

partone_table["judge_score_mean"] = partone_table["judge_scores"].apply(
    lambda scores: pd.Series(scores).dropna().mean()
)
partone_table_sorted = partone_table.sort_values("solver_difficulty_mean", ascending=False)

display(partone_table_sorted.head())

partone_table.to_csv(out_dir / f"safety_k{SOLVER_K}_partone_table_with_judge_score_mean.csv", index=False)
partone_table.to_json(out_dir / f"safety_k{SOLVER_K}_partone_table_with_judge_score_mean.json", orient="records", indent=2, force_ascii=False)
partone_table_sorted.to_csv(out_dir / f"safety_k{SOLVER_K}_partone_table_sorted.csv", index=False)
partone_table_sorted.to_json(out_dir / f"safety_k{SOLVER_K}_partone_table_sorted.json", orient="records", indent=2, force_ascii=False)


## Judge Score By Solver Prompt Difficulty Decile


In [ ]:
plot_df = partone_table.copy()
plot_df["difficulty_bin"] = pd.qcut(
    plot_df["solver_difficulty_mean"],
    q=10,
    labels=False,
    duplicates="drop",
)

bin_summary = (
    plot_df
    .groupby("difficulty_bin", observed=True)
    .agg(
        solver_difficulty_mean=("solver_difficulty_mean", "mean"),
        judge_score_mean=("judge_score_mean", "mean"),
        n_prompts=("prompt_id", "count"),
    )
    .reset_index()
)
bin_summary["difficulty_bin"] = bin_summary["difficulty_bin"] + 1

plt.figure(figsize=(8, 5))
plt.bar(bin_summary["difficulty_bin"], bin_summary["judge_score_mean"])
plt.xlabel("Solver prompt difficulty quantile bin")
plt.ylabel("Mean judge score")
plt.title("Mean judge score across solver prompt difficulty deciles")
plt.xticks(bin_summary["difficulty_bin"])
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(out_dir / f"safety_k{SOLVER_K}_difficulty_bin_summary.png", dpi=300, bbox_inches="tight")
plt.savefig(out_dir / f"safety_k{SOLVER_K}_difficulty_bin_summary.pdf", bbox_inches="tight")
plt.show()

display(bin_summary)

bin_summary.to_csv(out_dir / f"safety_k{SOLVER_K}_difficulty_bin_summary.csv", index=False)
bin_summary.to_json(out_dir / f"safety_k{SOLVER_K}_difficulty_bin_summary.json", orient="records", indent=2, force_ascii=False)


In [ ]:
# Per-judge-model view of Part 1.
# Each box shows the distribution across judge models within a solver difficulty bin.
model_plot_df = partone_table.copy()
model_plot_df["difficulty_bin"] = pd.qcut(
    model_plot_df["solver_difficulty_mean"],
    q=10,
    labels=False,
    duplicates="drop",
)
model_plot_df["difficulty_bin"] = model_plot_df["difficulty_bin"] + 1

judge_model_rows = []
for _, row in model_plot_df.iterrows():
    scores = row["judge_scores"]
    if not isinstance(scores, dict):
        continue
    for judge_model, score in scores.items():
        if score is None or pd.isna(score):
            continue
        judge_model_rows.append({
            "difficulty_bin": int(row["difficulty_bin"]),
            "prompt_id": row["prompt_id"],
            "judge_model": judge_model,
            "judge_score": float(score),
        })

judge_model_scores = pd.DataFrame(judge_model_rows)
judge_model_bin_summary = (
    judge_model_scores
    .groupby(["difficulty_bin", "judge_model"], observed=True)
    .agg(
        judge_score_mean=("judge_score", "mean"),
        n_prompts=("prompt_id", "count"),
    )
    .reset_index()
)

judge_model_bin_mean = (
    judge_model_bin_summary
    .groupby("difficulty_bin", observed=True)
    .agg(
        judge_score_mean=("judge_score_mean", "mean"),
        judge_score_sd=("judge_score_mean", "std"),
        n_judge_models=("judge_model", "count"),
    )
    .reset_index()
)

bins = sorted(judge_model_bin_summary["difficulty_bin"].unique())
box_values = [
    judge_model_bin_summary.loc[
        judge_model_bin_summary["difficulty_bin"] == bin_id,
        "judge_score_mean",
    ].dropna().to_numpy()
    for bin_id in bins
]

rng = np.random.default_rng(123)
plot_x = judge_model_bin_summary["difficulty_bin"].to_numpy(dtype=float)
plot_x = plot_x + rng.uniform(-0.14, 0.14, size=len(plot_x))

plt.figure(figsize=(9, 5.8))
plt.boxplot(
    box_values,
    positions=bins,
    widths=0.55,
    patch_artist=True,
    showfliers=False,
    boxprops={"facecolor": "0.88", "edgecolor": "0.35", "linewidth": 1.0},
    medianprops={"color": "0.2", "linewidth": 1.4},
    whiskerprops={"color": "0.45", "linewidth": 1.0},
    capprops={"color": "0.45", "linewidth": 1.0},
)
plt.scatter(
    plot_x,
    judge_model_bin_summary["judge_score_mean"],
    s=20,
    color="0.35",
    alpha=0.55,
    label="Judge model",
)
plt.plot(
    judge_model_bin_mean["difficulty_bin"],
    judge_model_bin_mean["judge_score_mean"],
    color="black",
    marker="D",
    markersize=5,
    linewidth=1.8,
    label="Mean across judge models",
)
plt.xlabel("Solver prompt difficulty quantile bin")
plt.ylabel("Mean judge score")
plt.title("Judge model score distribution across solver prompt difficulty deciles")
plt.xticks(bins)
plt.grid(axis="y", alpha=0.3)
plt.legend(loc="best")
plt.figtext(
    0.5,
    0.01,
    "Each box shows the distribution across judge models within a solver difficulty bin.",
    ha="center",
    fontsize=9,
)
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.savefig(out_dir / f"safety_k{SOLVER_K}_judge_model_bin_summary.png", dpi=300, bbox_inches="tight")
plt.savefig(out_dir / f"safety_k{SOLVER_K}_judge_model_bin_summary.pdf", bbox_inches="tight")
plt.show()

display(judge_model_bin_summary)
display(judge_model_bin_mean)

judge_model_bin_summary.to_csv(out_dir / f"safety_k{SOLVER_K}_judge_model_bin_summary.csv", index=False)
judge_model_bin_summary.to_json(out_dir / f"safety_k{SOLVER_K}_judge_model_bin_summary.json", orient="records", indent=2, force_ascii=False)
judge_model_bin_mean.to_csv(out_dir / f"safety_k{SOLVER_K}_judge_model_bin_summary_mean.csv", index=False)
judge_model_bin_mean.to_json(out_dir / f"safety_k{SOLVER_K}_judge_model_bin_summary_mean.json", orient="records", indent=2, force_ascii=False)


## Part Two: Solver Prompt Difficulty vs Judge Difficulty


In [ ]:
parttwo_table = paired_items[[
    "prompt_id",
    "source",
    "harmful_prompt",
    "solver_difficulty_mean",
    "solver_difficulty_laplace_se_mean",
    "solver_difficulty_sd_across_attacks",
    "solver_difficulty_min",
    "solver_difficulty_max",
    "solver_difficulty_range_low",
    "solver_difficulty_range_high",
    "judge_difficulty_centered",
    "judge_difficulty_centered_laplace_se",
    "solver_scores",
    "judge_scores",
]].copy()

parttwo_table_sorted = parttwo_table.sort_values("solver_difficulty_mean", ascending=False)

display(parttwo_table_sorted.head())

parttwo_table.to_csv(out_dir / f"safety_k{SOLVER_K}_parttwo_table_raw.csv", index=False)
parttwo_table.to_json(out_dir / f"safety_k{SOLVER_K}_parttwo_table_raw.json", orient="records", indent=2, force_ascii=False)
parttwo_table_sorted.to_csv(out_dir / f"safety_k{SOLVER_K}_parttwo_table_sorted.csv", index=False)
parttwo_table_sorted.to_json(out_dir / f"safety_k{SOLVER_K}_parttwo_table_sorted.json", orient="records", indent=2, force_ascii=False)


## Solver Prompt Difficulty vs Judge Difficulty Plot

Horizontal error bars show the min/max range of solver difficulty across attack families for the same prompt. Vertical error bars show judge-side Laplace SE.


In [ ]:
parttwo_difficulty_df = parttwo_table[[
    "prompt_id",
    "source",
    "solver_difficulty_mean",
    "solver_difficulty_laplace_se_mean",
    "solver_difficulty_sd_across_attacks",
    "solver_difficulty_min",
    "solver_difficulty_max",
    "solver_difficulty_range_low",
    "solver_difficulty_range_high",
    "judge_difficulty_centered",
    "judge_difficulty_centered_laplace_se",
]].dropna().copy()

raw_rho, raw_p = spearmanr(
    parttwo_difficulty_df["solver_difficulty_mean"],
    parttwo_difficulty_df["judge_difficulty_centered"],
)
print(f"Spearman rho: {raw_rho:.4f}")
print(f"p-value:      {raw_p:.4g}")

plt.figure(figsize=(6, 6))
plt.errorbar(
    parttwo_difficulty_df["solver_difficulty_mean"],
    parttwo_difficulty_df["judge_difficulty_centered"],
    xerr=[
        parttwo_difficulty_df["solver_difficulty_range_low"],
        parttwo_difficulty_df["solver_difficulty_range_high"],
    ],
    yerr=parttwo_difficulty_df["judge_difficulty_centered_laplace_se"],
    fmt="o",
    markersize=3,
    alpha=0.55,
    ecolor="gray",
    elinewidth=0.6,
    capsize=1.5,
)
plt.axhline(0, color="black", linewidth=1, alpha=0.35)
plt.axvline(0, color="black", linewidth=1, alpha=0.35)
plt.xlabel("Mean solver difficulty across attacks")
plt.ylabel("Judge difficulty centered")
plt.title(f"Solver vs Judge Difficulty (rho={raw_rho:.3f}, p={raw_p:.3g})")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(out_dir / f"safety_k{SOLVER_K}_solver_judge_difficulty_scatter.png", dpi=300, bbox_inches="tight")
plt.savefig(out_dir / f"safety_k{SOLVER_K}_solver_judge_difficulty_scatter.pdf", bbox_inches="tight")
plt.show()

display(parttwo_difficulty_df.head())

parttwo_difficulty_df.to_csv(out_dir / f"safety_k{SOLVER_K}_solver_judge_difficulty_scatter.csv", index=False)
parttwo_difficulty_df.to_json(out_dir / f"safety_k{SOLVER_K}_solver_judge_difficulty_scatter.json", orient="records", indent=2, force_ascii=False)


## Solver Percentile vs Judge Percentile


In [ ]:
percentile_df = parttwo_difficulty_df.copy()
percentile_df["solver_difficulty_percentile"] = percentile_df["solver_difficulty_mean"].rank(pct=True) * 100
percentile_df["judge_difficulty_percentile"] = percentile_df["judge_difficulty_centered"].rank(pct=True) * 100

pct_rho, pct_p = spearmanr(
    percentile_df["solver_difficulty_percentile"],
    percentile_df["judge_difficulty_percentile"],
)
print(f"Percentile Spearman rho: {pct_rho:.4f}")
print(f"Percentile p-value:      {pct_p:.4g}")

plt.figure(figsize=(6, 6))
plt.scatter(
    percentile_df["solver_difficulty_percentile"],
    percentile_df["judge_difficulty_percentile"],
    s=14,
    alpha=0.65,
)
plt.plot([0, 100], [0, 100], color="black", linewidth=1, alpha=0.4)
plt.xlabel("Solver prompt difficulty percentile")
plt.ylabel("Judge difficulty percentile")
plt.title(f"Solver vs Judge Difficulty Percentiles (rho={pct_rho:.3f}, p={pct_p:.3g})")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(out_dir / f"safety_k{SOLVER_K}_solver_judge_difficulty_percentiles.png", dpi=300, bbox_inches="tight")
plt.savefig(out_dir / f"safety_k{SOLVER_K}_solver_judge_difficulty_percentiles.pdf", bbox_inches="tight")
plt.show()

display(percentile_df.head())

percentile_df.to_csv(out_dir / f"safety_k{SOLVER_K}_solver_judge_difficulty_percentiles.csv", index=False)
percentile_df.to_json(out_dir / f"safety_k{SOLVER_K}_solver_judge_difficulty_percentiles.json", orient="records", indent=2, force_ascii=False)


## Attack-Family Detail Table

This table keeps the attack-expanded solver rows and attaches the prompt-level judge difficulty, for optional attack-family diagnostics.


In [ ]:
attack_family_detail = solver_attack_items.merge(
    judge_for_join[[
        "prompt_id",
        "judge_difficulty_centered",
        "judge_difficulty_centered_laplace_se",
        "judge_scores",
    ]],
    on="prompt_id",
    how="inner",
)

attack_family_detail = attack_family_detail.rename(columns={
    "difficulty_centered": "solver_attack_difficulty_centered",
    "difficulty_centered_laplace_se": "solver_attack_difficulty_centered_laplace_se",
})

print(f"attack_family_detail: {attack_family_detail.shape}")
display(attack_family_detail.head())

attack_family_detail.to_csv(out_dir / f"safety_k{SOLVER_K}_attack_family_detail.csv", index=False)
attack_family_detail.to_json(out_dir / f"safety_k{SOLVER_K}_attack_family_detail.json", orient="records", indent=2, force_ascii=False)
